In [91]:
import pandas as pd
import numpy as np
import re
import warnings
import csv
from sklearn.preprocessing import StandardScaler

**Load Datasets**

In [92]:
# ==========================================================
# FUNCTION: FIX MALFORMED CSV (WITHOUT DROPPING COLUMNS)
# ==========================================================
def fix_malformed_csv(filepath):
    print(f"\nProcessing: {filepath}")

    # Read everything as string first (to preserve original wording)
    df = pd.read_csv(filepath, encoding='utf-8', dtype=str)

    print(f"Initial shape: {df.shape}")

    # Detect rows where all data is crammed into first column
    malformed_rows = df[df.iloc[:, 1:].isnull().all(axis=1)]

    if len(malformed_rows) > 0:
        print(f"Found {len(malformed_rows)} malformed rows")

        fixed_rows = []

        for idx, row in malformed_rows.iterrows():
            full_string = row.iloc[0]

            # Proper CSV parsing (safer than split)
            parsed = list(csv.reader([full_string]))[0]

            # Ensure same number of columns
            if len(parsed) < len(df.columns):
                parsed += [None] * (len(df.columns) - len(parsed))
            elif len(parsed) > len(df.columns):
                parsed = parsed[:len(df.columns)]

            fixed_rows.append(parsed)

        fixed_df = pd.DataFrame(fixed_rows, columns=df.columns)

        # Replace malformed rows
        df.loc[malformed_rows.index] = fixed_df.values

        print("Malformed rows fixed successfully.")
    else:
        print("No malformed rows detected.")

    return df


# ==========================================================
# 1️⃣ LOAD ALL DATASETS
# ==========================================================

df1 = fix_malformed_csv(r'C:\Users\Marvella\YEAR 3\FYP\Exploratory Data Analysis\animal_disease_world_dataset(1).csv')
df2 = fix_malformed_csv(r'C:\Users\Marvella\YEAR 3\FYP\Exploratory Data Analysis\animal_disease_world_dataset(2).csv')
df3 = fix_malformed_csv(r'C:\Users\Marvella\YEAR 3\FYP\Exploratory Data Analysis\animal_disease_world_dataset(3).csv')
df4 = fix_malformed_csv(r'C:\Users\Marvella\YEAR 3\FYP\Exploratory Data Analysis\animal_disease_world_dataset(4).csv')

print("\n" + "="*80)
print("DATASET LOADING COMPLETE")
print("="*80)

print("Dataset 1 Shape:", df1.shape)
print("Dataset 2 Shape:", df2.shape)
print("Dataset 3 Shape:", df3.shape)
print("Dataset 4 Shape:", df4.shape)



Processing: C:\Users\Marvella\YEAR 3\FYP\Exploratory Data Analysis\animal_disease_world_dataset(1).csv
Initial shape: (431, 22)
No malformed rows detected.

Processing: C:\Users\Marvella\YEAR 3\FYP\Exploratory Data Analysis\animal_disease_world_dataset(2).csv
Initial shape: (40000, 24)
Found 17114 malformed rows
Malformed rows fixed successfully.

Processing: C:\Users\Marvella\YEAR 3\FYP\Exploratory Data Analysis\animal_disease_world_dataset(3).csv
Initial shape: (14000, 24)
Found 14000 malformed rows
Malformed rows fixed successfully.

Processing: C:\Users\Marvella\YEAR 3\FYP\Exploratory Data Analysis\animal_disease_world_dataset(4).csv
Initial shape: (30000, 24)
No malformed rows detected.

DATASET LOADING COMPLETE
Dataset 1 Shape: (431, 22)
Dataset 2 Shape: (40000, 24)
Dataset 3 Shape: (14000, 24)
Dataset 4 Shape: (30000, 24)


**Header Standardization**

*2.1 Standardize Column Names*

In [93]:
# ==========================================================
# STANDARDIZE COLUMN NAMES BEFORE MERGING
# ==========================================================

print("\nStandardizing column names...")

# Rename Animal_Type → Species in df1
if 'Animal_Type' in df1.columns:
    df1.rename(columns={'Animal_Type': 'Species'}, inplace=True)
    print("Renamed 'Animal_Type' to 'Species' in df1.")



Standardizing column names...
Renamed 'Animal_Type' to 'Species' in df1.


*2.2 Add Missing Columns to df1*

In [94]:
# Columns that should exist in final dataset
required_columns = list(df2.columns)

for col in required_columns:
    if col not in df1.columns:
        df1[col] = np.nan
        print(f"Added missing column '{col}' to df1.")


Added missing column 'Treatment' to df1.
Added missing column 'Description' to df1.


*2.3 Ensure Same Column Order*

In [95]:
# Reorder df1 columns to match df2, df3 and df4
df1 = df1[required_columns]

print("\ndf1 columns after alignment:")
print(df1.columns.tolist())

print("\ndf2 columns:")
print(df2.columns.tolist())

print("\ndf3 columns:")
print(df3.columns.tolist())

print("\ndf4 columns:")
print(df4.columns.tolist()) 



df1 columns after alignment:
['Species', 'Breed', 'Age', 'Gender', 'Weight', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Duration', 'Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing', 'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature', 'Heart_Rate', 'Disease_Prediction', 'Treatment', 'Description']

df2 columns:
['Species', 'Breed', 'Age', 'Gender', 'Weight', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Duration', 'Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing', 'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature', 'Heart_Rate', 'Disease_Prediction', 'Treatment', 'Description']

df3 columns:
['Species', 'Breed', 'Age', 'Gender', 'Weight', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Duration', 'Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing', 'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature', 'Hea

**Merge Datasets**

In [96]:
# ==========================================================
# 2️⃣ MERGE DATASETS SAFELY (NO COLUMN LOSS)
# ==========================================================

print("\n" + "="*80)
print("MERGING ALL DATASETS SAFELY")
print("="*80)

# Store all datasets in list
dataset_list = [df1, df2, df3, df4]

# Merge all
df = pd.concat(dataset_list, ignore_index=True, sort=False)

print(f"Merged Dataset Shape: {df.shape}")
print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")

print("\nColumns:")
print(df.columns.tolist())



MERGING ALL DATASETS SAFELY
Merged Dataset Shape: (84431, 24)
Total Rows: 84,431
Total Columns: 24

Columns:
['Species', 'Breed', 'Age', 'Gender', 'Weight', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Duration', 'Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing', 'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature', 'Heart_Rate', 'Disease_Prediction', 'Treatment', 'Description']


In [97]:
print("\n" + "="*80)
print("PREVIEW OF CLEANED DATASET (FIRST 10 ROWS)")
print("="*80)

# Make sure all columns are visible
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

print(df.head(10))



PREVIEW OF CLEANED DATASET (FIRST 10 ROWS)
  Species            Breed Age  Gender Weight          Symptom_1             Symptom_2          Symptom_3        Symptom_4 Duration Appetite_Loss Vomiting Diarrhea Coughing Labored_Breathing  \
0     Dog         Labrador   4    Male   25.0              Fever              Lethargy      Appetite Loss         Vomiting   3 days           Yes      Yes       No       No                No   
1     Cat          Siamese   2  Female    4.5           Coughing              Sneezing      Eye Discharge  Nasal Discharge   1 week            No       No       No      Yes                No   
2     Cow         Holstein   3  Female  600.0              Fever       Nasal Discharge  Labored Breathing         Coughing   5 days           Yes       No       No      Yes               Yes   
3     Dog           Beagle   1    Male   10.0           Diarrhea              Vomiting           Lethargy    Appetite Loss   2 days           Yes      Yes      Yes       No        

In [98]:
print("\nColumns in Dataset:")
print(df.columns.tolist())
print(f"\nTotal Columns: {len(df.columns)}")



Columns in Dataset:
['Species', 'Breed', 'Age', 'Gender', 'Weight', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Duration', 'Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing', 'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature', 'Heart_Rate', 'Disease_Prediction', 'Treatment', 'Description']

Total Columns: 24


In [99]:
print("\nRandom Sample Rows:")
print(df.sample(5))


Random Sample Rows:
       Species         Breed Age  Gender  Weight           Symptom_1                    Symptom_2            Symptom_3              Symptom_4 Duration Appetite_Loss Vomiting Diarrhea Coughing  \
53965  Dolphin    Bottlenose  15  Female   536.1  Erythematous spots  Localized hyperpigmentation  Fragile hair shafts      Circular alopecia  30 days           Yes      Yes       No       No   
83409     Bird    Bird Breed   8  Female  719.03    Patchy hair loss              Skin irritation     Inflamed lesions  Persistent scratching  19 days           Yes       No      Yes      Yes   
38449    Horse  Thoroughbred  11    Male  170.95                Pain                     Sneezing         Skin Lesions                 Anemia   5 days           Yes       No      Yes       No   
83831     Goat        Alpine   3    Male  570.32    Patchy hair loss              Skin irritation     Inflamed lesions  Persistent scratching   7 days           Yes      Yes       No      Yes   
5542

**Fix Text Encoding**

In [100]:
print("\nFixing text encoding issues...")

def fix_text_encoding(text):
    if isinstance(text, str):
        try:
            return text.encode('latin1').decode('utf-8')
        except:
            return text
    return text

# Columns likely to contain long medical text
text_columns = ['Treatment', 'Description', 'Disease', 'Disease_Prediction']

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(fix_text_encoding)
        print(f"Encoding fixed in column: {col}")


Fixing text encoding issues...
Encoding fixed in column: Treatment
Encoding fixed in column: Description
Encoding fixed in column: Disease_Prediction


**Clean Body Temperature**

Remove °C symbol and convert to numeric values

In [101]:
if 'Body_Temperature' in df.columns:
    print("\nCleaning Body_Temperature column...")

    df['Body_Temperature'] = (
        df['Body_Temperature']
        .astype(str)
        .str.replace('°C', '', regex=False)
        .str.strip()
    )

    df['Body_Temperature'] = pd.to_numeric(df['Body_Temperature'], errors='coerce')

    print("Missing Body_Temperature values:", df['Body_Temperature'].isnull().sum())

    # Fill missing with median
    df['Body_Temperature'].fillna(df['Body_Temperature'].median(), inplace=True)


Cleaning Body_Temperature column...
Missing Body_Temperature values: 0


**Convert Yes/No --> Binary**

In [102]:
print("\nConverting Yes/No columns to binary...")

yes_no_columns = []

for col in df.columns:
    if df[col].dtype == 'object':
        unique_vals = df[col].dropna().unique()
        if set(unique_vals).issubset({'Yes', 'No'}):
            yes_no_columns.append(col)

print("Yes/No Columns Found:", yes_no_columns)

for col in yes_no_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0})


Converting Yes/No columns to binary...
Yes/No Columns Found: ['Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing', 'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge']


**Species-Based Weight Correction**

*Clean Species Column*

In [103]:
if 'Species' in df.columns:
    print("\nCleaning Species column...")

    df['Species'] = (
        df['Species']
        .astype(str)
        .str.replace(r'[^A-Za-z\s]', '', regex=True)  # Remove numbers & special characters
        .str.strip()
        .str.title()
    )

    print("Species cleaned successfully.")
    print("Unique Species:", df['Species'].unique())
    print("\n")
    print("Unique Breeds:", df['Breed'].unique())


Cleaning Species column...
Species cleaned successfully.
Unique Species: ['Dog' 'Cat' 'Cow' 'Horse' 'Rabbit' 'Sheep' 'Goat' 'Pig' 'Bird' 'Fish'
 'Shark' 'Lizard' 'Snake' 'Turtle' 'Dolphin' 'Frog']


Unique Breeds: ['Labrador' 'Siamese' 'Holstein' 'Beagle' 'Persian' 'Thoroughbred'
 'German Shepherd' 'Maine Coon' 'Bulldog' 'Jersey' 'Poodle' 'Bengal'
 'Arabian' 'Hereford' 'Chihuahua' 'Ragdoll' 'Siberian Husky' 'Clydesdale'
 'British Shorthair' 'Angus' 'Golden Retriever' 'Russian Blue'
 'Quarter Horse' 'Appaloosa' 'Dachshund' 'Sphynx' 'Abyssinian' 'Brahman'
 'Husky' 'Morgan' 'Limousin' 'Border Collie' 'Devon Rex' 'Standardbred'
 'Red Angus' 'Shire' 'Guernsey' 'Labrador Retriever' 'Scottish Fold'
 'Simmental' 'Rottweiler' 'Siberian' 'American Quarter' 'Shorthorn'
 'Shih Tzu' 'American Curl' 'Percheron' 'Manx' 'Tennessee Walker'
 'Dalmatian' 'Bombay' 'Welsh Pony' 'Brown Swiss' 'Akita' 'Boxer'
 'Holland Lop' 'Merino' 'Boer' 'Yorkshire' 'Mini Rex' 'Suffolk' 'Alpine'
 'Duroc' 'Corgi' 'Burmese'

*Convert Weight to Numeric*

In [104]:
if 'Weight' in df.columns:
    print("\nConverting Weight to numeric...")

    df['Weight'] = (
        df['Weight']
        .astype(str)
        .str.replace('kg', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )

    df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')

    print("Missing Weight values:", df['Weight'].isnull().sum())



Converting Weight to numeric...
Missing Weight values: 0


*Define Realistics Biological Weight Ranges*

In [105]:
species_weight_ranges = {
    'Dog': (1, 90),
    'Cat': (2, 10),
    'Cow': (200, 1200),
    'Horse': (300, 1000),
    'Rabbit': (1, 10),
    'Sheep': (20, 150),
    'Goat': (15, 120),
    'Pig': (50, 350),
    'Bird': (0.05, 5),
    'Fish': (0.01, 100),
    'Shark': (100, 1100),
    'Lizard': (0.01, 20),
    'Snake': (0.1, 80),
    'Turtle': (0.1, 300),
    'Dolphin': (100, 300),
    'Frog': (0.01, 5)
}

In [106]:
breed_weight_ranges = {
    # --- DOGS ---
    'Labrador': (25, 36), 'Labrador Retriever': (25, 36), 'Beagle': (9, 15), 
    'German Shepherd': (30, 40), 'Bulldog': (18, 25), 'Poodle': (20, 32), 
    'Chihuahua': (1, 3), 'Siberian Husky': (16, 27), 'Husky': (16, 27),
    'Golden Retriever': (25, 35), 'Dachshund': (5, 12), 'Border Collie': (12, 20),
    'Rottweiler': (35, 60), 'Shih Tzu': (4, 7), 'Dalmatian': (15, 32),
    'Akita': (32, 59), 'Boxer': (25, 32), 'Corgi': (10, 14), 
    'Doberman Pinscher': (32, 45), 'Pit Bull': (14, 36), 'Cocker Spaniel': (11, 15),
    'Yorkshire Terrier': (2, 4),

    # --- CATS ---
    'Siamese': (3, 5), 'Persian': (3, 5), 'Maine Coon': (5, 10), 'Bengal': (4, 8),
    'Ragdoll': (4, 9), 'British Shorthair': (4, 8), 'Russian Blue': (3, 5),
    'Sphynx': (3, 5), 'Abyssinian': (3, 5), 'Devon Rex': (2, 4), 
    'Scottish Fold': (3, 6), 'Siberian': (4, 8), 'American Curl': (3, 5),
    'Manx': (4, 6), 'Bombay': (3, 5), 'Burmese': (3, 5), 'Himalayan': (3, 5),

    # --- HORSES / PONIES ---
    'Thoroughbred': (400, 600), 'Arabian': (360, 500), 'Clydesdale': (800, 1000),
    'Quarter Horse': (450, 550), 'American Quarter': (450, 550), 'Appaloosa': (450, 550),
    'Morgan': (400, 500), 'Standardbred': (450, 550), 'Shire': (850, 1100),
    'Percheron': (800, 1000), 'Tennessee Walker': (450, 550), 'Tennessee Walking Horse': (450, 550),
    'Welsh Pony': (200, 350), 'Shetland Pony': (100, 200), 'Paint': (430, 540),
    'Belgian': (800, 1000), 'Mustang': (350, 500), 'Andalusian': (450, 580), 'Pinto': (400, 550),

    # --- CATTLE ---
    'Holstein': (550, 700), 'Jersey': (350, 500), 'Hereford': (550, 800), 
    'Angus': (500, 800), 'Aberdeen Angus': (500, 800), 'Red Angus': (500, 800),
    'Brahman': (450, 900), 'Limousin': (550, 900), 'Guernsey': (400, 600),
    'Simmental': (700, 1000), 'Shorthorn': (600, 900), 'Brown Swiss': (600, 900),
    'Charolais': (700, 1100), 'Dexter': (300, 450), 'Ayrshire': (450, 600),
    'Red Poll': (500, 700), 'Belted Galloway': (400, 600),

    # --- SHEEP / GOATS ---
    'Merino': (55, 100), 'Boer': (80, 130), 'Suffolk': (80, 150), 'Alpine': (60, 80),
    'Dorper': (70, 100), 'Nubian': (60, 90), 'Cheviot': (50, 85), 'Toggenburg': (55, 75),
    'Romney': (70, 110), 'Kiko': (50, 80), 'Lincoln': (100, 160), 'LaMancha': (55, 75),
    'Texel': (70, 120), 'Finnsheep': (55, 85), 'Saanen': (60, 85), 'Leicester Longwool': (80, 110),
    'Nigerian Dwarf': (20, 35), 'Dorset': (70, 110), 'Karakul': (45, 80), 'Angora': (35, 60),
    'Corriedale': (60, 100), 'Southdown': (60, 100), 'Tunis': (60, 100), 'Rambouillet': (70, 120),

    # --- PIGS ---
    'Yorkshire': (230, 370), 'Duroc': (230, 380), 'Berkshire': (250, 350),
    'Tamworth': (230, 350), 'Poland China': (250, 400), 'Landrace': (200, 300),
    'Wessex Saddleback': (250, 320), 'Large White': (250, 380), 'Hampshire': (230, 300),
    'Pietrain': (200, 270), 'Chester White': (230, 350),

    # --- RABBITS ---
    'Holland Lop': (1.0, 1.8), 'Mini Rex': (1.4, 2.0), 'Flemish Giant': (6.0, 10.0),
    'English Angora': (2.3, 3.5), 'Dutch': (1.5, 2.5), 'Mini Lop': (1.4, 2.7),
    'English Spot': (2.3, 3.6), 'English Lop': (4.0, 5.0), 'Netherland Dwarf': (0.5, 1.1),

    # --- EXOTICS / BIRDS / AQUATIC / REPTILES ---
    'Parrot': (0.3, 1.0), 'Goldfish': (0.1, 0.3), 'Great White': (600, 2100),
    'Hammerhead': (200, 500), 'Iguana': (4, 9), 'Green Iguana': (4, 9),
    'Cobra': (1.5, 6), 'Red-eared Slider': (0.5, 1.5), 'Bottlenose': (150, 600),
    'Eagle': (3, 7), 'Gecko': (0.05, 0.1), 'Leopard Gecko': (0.05, 0.1),
    'Tree Frog': (0.02, 0.05), 'Clownfish': (0.1, 0.2), 'Python': (30, 100),
    'Bearded Dragon': (0.3, 0.5), 'Snake Breed': (1, 10), 'Fish Breed': (0.1, 5), 
    'Bird Breed': (0.1, 2)
}

In [107]:
# Clean Species Column
if 'Species' in df.columns:
    print("\nCleaning Species column...")
    df['Species'] = (
        df['Species']
        .astype(str)
        .str.replace(r'[^A-Za-z\s]', '', regex=True)
        .str.strip()
        .str.title()
    )
    print("Species cleaned successfully.")
    print("Unique Species:", df['Species'].unique())
    print("\nUnique Breeds:", df['Breed'].unique())

# Convert Weight to numeric
if 'Weight' in df.columns:
    print("\nConverting Weight to numeric...")
    df['Weight'] = (
        df['Weight']
        .astype(str)
        .str.replace('kg', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')
    print("Missing Weight values:", df['Weight'].isnull().sum())


Cleaning Species column...
Species cleaned successfully.
Unique Species: ['Dog' 'Cat' 'Cow' 'Horse' 'Rabbit' 'Sheep' 'Goat' 'Pig' 'Bird' 'Fish'
 'Shark' 'Lizard' 'Snake' 'Turtle' 'Dolphin' 'Frog']

Unique Breeds: ['Labrador' 'Siamese' 'Holstein' 'Beagle' 'Persian' 'Thoroughbred'
 'German Shepherd' 'Maine Coon' 'Bulldog' 'Jersey' 'Poodle' 'Bengal'
 'Arabian' 'Hereford' 'Chihuahua' 'Ragdoll' 'Siberian Husky' 'Clydesdale'
 'British Shorthair' 'Angus' 'Golden Retriever' 'Russian Blue'
 'Quarter Horse' 'Appaloosa' 'Dachshund' 'Sphynx' 'Abyssinian' 'Brahman'
 'Husky' 'Morgan' 'Limousin' 'Border Collie' 'Devon Rex' 'Standardbred'
 'Red Angus' 'Shire' 'Guernsey' 'Labrador Retriever' 'Scottish Fold'
 'Simmental' 'Rottweiler' 'Siberian' 'American Quarter' 'Shorthorn'
 'Shih Tzu' 'American Curl' 'Percheron' 'Manx' 'Tennessee Walker'
 'Dalmatian' 'Bombay' 'Welsh Pony' 'Brown Swiss' 'Akita' 'Boxer'
 'Holland Lop' 'Merino' 'Boer' 'Yorkshire' 'Mini Rex' 'Suffolk' 'Alpine'
 'Duroc' 'Corgi' 'Burmese' 

*Detect Unrealistic Weights*

In [108]:
def is_unrealistic(row):
    species = str(row.get('Species', '')).strip()
    breed   = str(row.get('Breed', '')).strip()
    weight = row.get('Weight')
    
    try:
        weight = float(weight)
    except (TypeError, ValueError):
        return True

    # 1. Prefer breed range if available
    if breed in breed_weight_ranges:
        min_w, max_w = breed_weight_ranges[breed]
    # 2. Otherwise fall back to species range
    elif species in species_weight_ranges:
        min_w, max_w = species_weight_ranges[species]
    else:
        return True  # unknown species → unrealistic

    return not (min_w <= weight <= max_w)

df['Unrealistic_Weight'] = df.apply(is_unrealistic, axis=1)
print("\nUnrealistic weight count:", df['Unrealistic_Weight'].sum())


Unrealistic weight count: 75739


In [109]:
def generate_realistic_weight(row):
    species = str(row.get('Species', '')).strip()
    breed   = str(row.get('Breed', '')).strip()
    
    if breed in breed_weight_ranges:
        min_w, max_w = breed_weight_ranges[breed]
    elif species in species_weight_ranges:
        min_w, max_w = species_weight_ranges[species]
    else:
        return np.nan
    
    return round(np.random.uniform(min_w, max_w), 2)

# Apply correction only to unrealistic or missing weights
df.loc[df['Unrealistic_Weight'] == True, 'Weight'] = \
    df[df['Unrealistic_Weight'] == True].apply(generate_realistic_weight, axis=1)

print("Unrealistic/missing weights corrected using breed‑aware random values.")

Unrealistic/missing weights corrected using breed‑aware random values.


*Remove Helper Column*

In [110]:
# Remove helper column
df.drop(columns=['Unrealistic_Weight'], inplace=True)

# Preview updated weights
print("\nPreview of corrected weights:")
print(df[['Species', 'Breed', 'Weight']].sample(10))


Preview of corrected weights:
      Species             Breed  Weight
3551   Lizard             Gecko    0.08
11395     Cat           Persian    3.52
15988  Turtle  Red-eared Slider    0.80
35046   Shark        Hammerhead  222.67
73432  Lizard      Green Iguana    5.91
44431    Bird            Parrot    0.65
52305     Dog          Labrador   34.11
52633     Cow          Holstein  589.68
67439    Goat            Alpine   77.28
26137    Bird             Eagle    5.19


**Remove Specific Rows in Disease_Prediction Row**

*Remove Uncessary Rows for Disease_Prediction*

In [111]:
# Remove rows where 'Disease_Prediction' contains "Subtype--"
if 'Disease_Prediction' in df.columns:
    initial_count = len(df)
    df = df[~df['Disease_Prediction'].str.contains('Subtype--', na=False)]
    removed_count = initial_count - len(df)
    print(f"Removed {removed_count} rows containing 'Subtype--' in Disease_Prediction.")

Removed 171 rows containing 'Subtype--' in Disease_Prediction.


*Rule-Based Filtering*

In [112]:
disease_counts = df['Disease_Prediction'].value_counts()
print("\n=== Disease Distribution ===")
for disease, count in disease_counts.items():
    print(f"{disease} - {count}")


=== Disease Distribution ===
Parasitic Infestation - 8941
Septicemia - 8932
Bacterial Pneumonia - 8868
Respiratory Distress Syndrome - 4673
Viral Infection - 4602
Gastroenteritis - 4586
Dermatitis - 4539
Dermatophytosis - 4526
Viral Respiratory Disease - 4342
Rabbit Ear Mite Infestation  - 664
Canine Atopic Dermatitis  - 638
Rabbit Hemorrhagic Disease  - 636
Lizard Dermal Fungal Infection  - 636
Lizard Parasitic Infestation  - 634
Avian Respiratory Mycoplasmosis  - 633
Ichthyophthirius Multifiliis Infection  - 633
Fin Rot Bacterial Disease  - 631
Swim Bladder Disorder  - 629
Equine Gastric Ulcer Syndrome  - 622
Equine Laminitis Disorder  - 621
Bovine Mastitis  - 619
Reptile Vitamin A Deficiency  - 614
Fish Gill Fluke Infestation  - 610
Caprine Pneumonia Syndrome  - 606
Goat Parasitic Enteritis  - 604
Canine Parvoviral Enteritis  - 604
Snake Inclusion Body Disease  - 603
Feline Infectious Peritonitis  - 603
Avian Nutritional Deficiency Syndrome  - 602
Equine Influenza Virus Infection  

In [113]:
# Define allowed diseases per species
species_disease_map = {
    'Dog': [
        'Canine Atopic Dermatitis', 'Canine Parvoviral Enteritis', 'Canine Hip Dysplasia Syndrome',
        'Canine Infectious Tracheobronchitis', 'Canine Distemper Virus Infection', 'Canine Parvovirus',
        'Canine Distemper', 'Kennel Cough', 'Canine Leptospirosis', 'Canine Hepatitis', 
        'Canine Flu', 'Canine Cough', 'Canine Heartworm Disease', 'Canine Influenza', 
        'Canine Infectious Hepatitis', 'Lyme Disease', 'Leptospirosis', 'Dermatitis',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia', 'Viral Infection', 
        'Gastroenteritis', 'Dermatophytosis'
    ],
    'Cat': [
        'Feline Infectious Peritonitis', 'Feline Chronic Kidney Disease', 'Feline Hyperthyroidism',
        'Feline Upper Respiratory Infection', 'Feline Dermatophytosis', 'Feline Calicivirus',
        'Feline Herpesvirus', 'Feline Leukemia Virus', 'Feline Asthma', 'Feline Panleukopenia',
        'Feline Rhinotracheitis', 'Feline Respiratory Disease Complex', 'Feline Viral Rhinotracheitis',
        'Feline Immunodeficiency Virus', 'Feline Chlamydiosis', 'Feline Respiratory Infection',
        'Feline Chlamydia', 'Feline Coronavirus', 'Feline Renal Disease', 'Hyperthyroidism',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia', 'Viral Infection', 
        'Gastroenteritis', 'Dermatophytosis'
    ],
    'Cow': [
        'Bovine Mastitis', 'Bovine Ketosis Disorder', 'Bovine Respiratory Disease Complex',
        'Bovine Parasitic Gastroenteritis', 'Bovine Tuberculosis', 'Bovine Respiratory Disease',
        'Bovine Johne\'s Disease', 'Bovine Viral Diarrhea', 'Bovine Respiratory Syncytial Virus',
        'Bovine Influenza', 'Bovine Coccidiosis', 'Bovine Leukemia Virus', 'Bovine Pneumonia',
        'Bovine Parainfluenza', 'Foot and Mouth Disease', 'Johne\'s Disease', 'Mastitis',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia', 'Viral Infection'
    ],
    'Horse': [
        'Equine Gastric Ulcer Syndrome', 'Equine Laminitis Disorder', 'Equine Influenza Virus Infection',
        'Equine Colic Syndrome', 'Equine Protozoal Myeloencephalitis', 'Equine Influenza',
        'Equine Infectious Anemia', 'Equine Herpesvirus', 'Equine Arthritis', 'Equine Laminitis',
        'Equine Encephalomyelitis', 'Equine Rhinopneumonitis', 'Equine Metabolic Syndrome',
        'Equine Encephalitis', 'Equine Leptospirosis', 'Equine Osteoarthritis', 'Equine Cushing\'s Disease',
        'Equine Viral Arteritis', 'Equine Piroplasmosis', 'Equine Pneumonia', 'Equine Lyme Disease',
        'Equine West Nile Virus', 'Strangles', 'Laminitis', 'Colic',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia'
    ],
    'Rabbit': [
        'Rabbit Ear Mite Infestation', 'Rabbit Hemorrhagic Disease', 'Rabbit Dental Malocclusion Syndrome',
        'Rabbit Gastrointestinal Stasis', 'Rabbit Pasteurellosis Infection', 'Myxomatosis',
        'Rabbit Viral Hemorrhagic Disease', 'Rabbit Syphilis', 'Rabbit Calicivirus', 'Snuffles',
        'Gastrointestinal Stasis', 'Viral Hemorrhagic Disease',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia', 'Coccidiosis'
    ],
    'Goat': [
        'Caprine Pneumonia Syndrome', 'Goat Parasitic Enteritis', 'Caprine Caseous Lymphadenitis',
        'Goat Coccidiosis Infection', 'Caprine Arthritis Encephalitis', 'Caprine Pleuropneumonia',
        'Caprine Respiratory Disease', 'Goat Pox', 'Caprine Arthritis', 'Caprine Viral Arthritis',
        'Caprine Arthritis Encephalitis Virus', 'Contagious Ecthyma', 'Caseous Lymphadenitis',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia', 'Coccidiosis'
    ],
    'Sheep': [
        'Scrapie', 'Bluetongue', 'Chlamydia in Sheep', 'Maedi-Visna', 'Footrot', 
        'Scrapie Disease', 'Blue Tongue Virus', 'Bluetongue Virus', 'Blue Tongue Disease',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia'
    ],
    'Pig': [
        'Swine Influenza', 'Actinobacillus Pleuropneumonia', 'Porcine Epidemic Diarrhea',
        'Swine Fever', 'Swine Dysentery', 'Porcine Respiratory Disease Complex', 'Swine Flu',
        'African Swine Fever', 'Porcine Reproductive and Respiratory Syndrome', 'Actinobacillus Suis',
        'Porcine Circovirus Disease', 'Porcine Epidemic Diarrhea Virus', 'Swine Erysipelas',
        'Foot and Mouth Disease', 'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia'
    ],
    'Bird': [
        'Avian Respiratory Mycoplasmosis', 'Avian Nutritional Deficiency Syndrome',
        'Psittacine Beak and Feather Disease', 'Avian Aspergillosis', 'Avian Influenza Infection',
        'Avian Influenza', 'Respiratory Infection', 'Viral Infection',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia'
    ],
    'Fish': [
        'Ichthyophthirius Multifiliis Infection', 'Fin Rot Bacterial Disease', 'Swim Bladder Disorder',
        'Fish Gill Fluke Infestation', 'Columnaris Bacterial Infection', 'Fungal Infection',
        'Parasitic Infestation', 'Septicemia'
    ],
    'Lizard': [
        'Lizard Dermal Fungal Infection', 'Lizard Parasitic Infestation', 'Reptilian Respiratory Infection',
        'Reptilian Metabolic Bone Disease', 'Reptile Vitamin A Deficiency', 'Fungal Infection',
        'Parasitic Infestation', 'Septicemia'
    ],
    'Snake': [
        'Snake Inclusion Body Disease', 'Snake Respiratory Infection', 'Ophidian Dermatitis Syndrome',
        'Snake Parasitic Mite Infestation', 'Snake Gastrointestinal Impaction', 'Reptilian Respiratory Infection',
        'Parasitic Infestation', 'Septicemia'
    ],
    'Turtle': [
        'Reptilian Respiratory Infection', 'Reptilian Metabolic Bone Disease', 'Reptile Vitamin A Deficiency',
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia'
    ],
    'Shark': [
        'Parasitic Infestation', 'Septicemia', 'Bacterial Pneumonia', 'Viral Infection'
    ],
    'Dolphin': [
        'Respiratory Distress Syndrome', 'Viral Respiratory Disease', 'Septicemia', 
        'Bacterial Pneumonia', 'Parasitic Infestation'
    ],
    'Frog': [
        'Fungal Infection', 'Septicemia', 'Parasitic Infestation', 'Bacterial Pneumonia'
    ]
}

In [114]:
# 1. Standardize the Dictionary for comparison (Strip and Lowercase)
clean_map = {k: [d.strip().lower() for d in v] for k, v in species_disease_map.items()}

# 2. Define the check function (Handles potential NaNs and Case sensitivity)
def check_alignment(row):
    species = str(row['Species']).strip()
    # Correcting the column name to match your previous output 'Disease_Prediction'
    disease = str(row['Disease_Prediction']).strip().lower()
    
    allowed_diseases = clean_map.get(species, [])
    return disease in allowed_diseases

# Create boolean mask
df['is_correct'] = df.apply(check_alignment, axis=1)

# Summary
correct_count = df['is_correct'].sum()
incorrect_count = len(df) - correct_count

print("=== Species-Disease Alignment Check ===")
print(f"Total rows: {len(df)}")
print(f"Correctly aligned rows: {correct_count}")
print(f"Misaligned rows: {incorrect_count}")
print(f"Percentage correct: {correct_count / len(df) * 100:.2f}%")

=== Species-Disease Alignment Check ===
Total rows: 84260
Correctly aligned rows: 59320
Misaligned rows: 24940
Percentage correct: 70.40%


In [115]:
# --- Final Drop of Misaligned Rows ---

# 1. Identify initial count for the report
initial_count = len(df)

# 2. Keep only the rows where is_correct is True
# This effectively 'drops' all rows where is_correct was False
df = df[df['is_correct'] == True].reset_index(drop=True)

# 3. Remove the helper column
if 'is_correct' in df.columns:
    df.drop(columns=['is_correct'], inplace=True)

# 4. Final Audit Output
rows_dropped = initial_count - len(df)
print("===== CLEANING COMPLETE =====")
print(f"Total Rows Dropped: {rows_dropped}")
print(f"Final Dataset Shape: {df.shape}")
print("==============================")

===== CLEANING COMPLETE =====
Total Rows Dropped: 24940
Final Dataset Shape: (59320, 24)


**Handle Missing Values**

In [116]:
print("\nHandling missing values...")

for col in df.columns:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        if col in ['Treatment', 'Description']:

            # Fill Treatment and Description explicitly
            df[col].fillna("Not_Recorded", inplace=True)
            print(f"Filled {col} with 'Not_Recorded'")

        elif df[col].dtype in ['float64', 'int64']:

            # Numeric columns → fill with median
            df[col].fillna(df[col].median(), inplace=True)
            print(f"Filled {col} with median")

        else:

            # Other categorical columns → fill with mode
            mode_value = df[col].mode()
            if len(mode_value) > 0:
                df[col].fillna(mode_value[0], inplace=True)
                print(f"Filled {col} with mode")
                
            else:
                df[col].fillna("Unknown", inplace=True)
                print(f"Filled {col} with 'Unknown'")


Handling missing values...
Filled Treatment with 'Not_Recorded'
Filled Description with 'Not_Recorded'


**Remove Duplicates**

In [117]:
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates removed.")

print("Final shape after duplicate removal:", df.shape)


Duplicate rows found: 5
Duplicates removed.
Final shape after duplicate removal: (59315, 24)


**Outlier Treatment**

In [118]:
print("\nDetecting and capping outliers using IQR method...")

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"{col}: {len(outliers)} outliers detected")

    # Cap values
    df[col] = np.clip(df[col], lower_bound, upper_bound)

print("Outliers capped for all numerical features.")


Detecting and capping outliers using IQR method...
Weight: 1433 outliers detected
Appetite_Loss: 0 outliers detected
Vomiting: 0 outliers detected
Diarrhea: 0 outliers detected
Coughing: 0 outliers detected
Labored_Breathing: 0 outliers detected
Lameness: 0 outliers detected
Skin_Lesions: 0 outliers detected
Nasal_Discharge: 0 outliers detected
Eye_Discharge: 0 outliers detected
Body_Temperature: 0 outliers detected
Outliers capped for all numerical features.


**Feature Engineering**

In [119]:
print("\nStarting Feature Engineering: Dangerous Level / Risk Score")

# 1️. Count Total Symptoms
# Replace the column names with your dataset's symptom columns
symptom_cols = [
    'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing',
    'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge'
]

# Ensure these columns exist
existing_symptom_cols = [col for col in symptom_cols if col in df.columns]

# Sum of symptoms
df['Symptom_Count'] = df[existing_symptom_cols].sum(axis=1)
print("✓ Symptom_Count feature created.")

# 2️. Temperature Risk (0-1 scale)
# Baseline: 37°C normal, 42°C extremely high
df['Temp_Risk'] = df['Body_Temperature'].apply(
    lambda x: max(0, min((x - 37) / (42 - 37), 1))
)
print("✓ Temp_Risk feature created.")

# 3️. Symptom Risk (0-1 scale)
max_symptoms = len(existing_symptom_cols)
df['Symptom_Risk'] = df['Symptom_Count'] / max_symptoms
print("✓ Symptom_Risk feature created.")

# 4️. Critical Symptoms Risk (weighted higher)
critical_cols = ['Labored_Breathing', 'Lameness']
existing_critical_cols = [col for col in critical_cols if col in df.columns]
df['Critical_Risk'] = df[existing_critical_cols].sum(axis=1) / len(existing_critical_cols) if existing_critical_cols else 0
print("✓ Critical_Risk feature created.")

# 5. Combine into Dangerous_Level (%)
# Weighted combination: Temperature 50%, Symptoms 30%, Critical 20%
df['Dangerous_Level'] = (
    0.5 * df['Temp_Risk'] +
    0.3 * df['Symptom_Risk'] +
    0.2 * df['Critical_Risk']
) * 100  # Convert to percentage
print("✓ Dangerous_Level (%) feature created.")

# 6️. Optional: Categorize into Low/Moderate/High/Critical
def danger_category(x):
    if x < 25:
        return 'Low'
    elif x < 50:
        return 'Moderate'
    elif x < 75:
        return 'High'
    else:
        return 'Critical'

df['Danger_Level_Category'] = df['Dangerous_Level'].apply(danger_category)
print("✓ Danger_Level_Category feature created.")

# 7️⃣ Preview the new features
preview_cols = ['Body_Temperature', 'Symptom_Count', 'Temp_Risk',
                'Symptom_Risk', 'Critical_Risk', 'Dangerous_Level', 'Danger_Level_Category']

print("\nPreview of Feature Engineered Columns:")
print(df[preview_cols].head(10))



Starting Feature Engineering: Dangerous Level / Risk Score
✓ Symptom_Count feature created.
✓ Temp_Risk feature created.
✓ Symptom_Risk feature created.
✓ Critical_Risk feature created.
✓ Dangerous_Level (%) feature created.
✓ Danger_Level_Category feature created.

Preview of Feature Engineered Columns:
   Body_Temperature  Symptom_Count  Temp_Risk  Symptom_Risk  Critical_Risk  Dangerous_Level Danger_Level_Category
0              40.1              3       0.62         0.375            0.5            52.25                  High
1              39.2              2       0.44         0.250            0.0            29.50              Moderate
2              39.8              3       0.56         0.375            0.5            49.25              Moderate
3              39.3              2       0.46         0.250            0.5            40.50              Moderate
4              40.0              3       0.60         0.375            0.5            51.25                  High
5        

**Encode Cateforical Variables**

In [120]:
categorical_cols = df.select_dtypes(include=['object']).columns

print("\nEncoding categorical variables...")

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Shape after encoding:", df_encoded.shape)



Encoding categorical variables...
Shape after encoding: (59315, 1078)


**Feature Scaling**

In [121]:
from sklearn.preprocessing import StandardScaler

print("Identifying numerical columns for standardization...")

scaler = StandardScaler()

# Get all numerical columns in the encoded dataset
numerical_cols_all = df_encoded.select_dtypes(include=['float32', 'float64', 'int32', 'int64']).columns

# Exclude Dangerous_Level from scaling to keep its original 0-100 interpretation
exclude_from_scaling = ['Dangerous_Level']
cols_to_scale = [c for c in numerical_cols_all if c not in exclude_from_scaling]

print(f"Found {len(cols_to_scale)} numerical features to scale (excluding {exclude_from_scaling}).")

df_encoded[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])

print("Data standardized successfully.")
print("Note: 'Dangerous_Level' was NOT scaled (remains in original 0-100 range).")

Identifying numerical columns for standardization...
Found 15 numerical features to scale (excluding ['Dangerous_Level']).
Data standardized successfully.
Note: 'Dangerous_Level' was NOT scaled (remains in original 0-100 range).


**Round Specific Columns to 2 Decimal Places**

In [122]:
print("\nRounding 'Weight', 'Temp_Risk', and 'Dangerous_Level' to 2 decimal places...")

for col in ['Weight', 'Temp_Risk', 'Dangerous_Level']:
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].round(2)
        print(f"Rounded {col} (in final encoded dataset).")

# Also round in original df (if you want to save both, but we'll save df_encoded)
for col in ['Weight', 'Temp_Risk', 'Dangerous_Level']:
    if col in df.columns:
        df[col] = df[col].round(2)



Rounding 'Weight', 'Temp_Risk', and 'Dangerous_Level' to 2 decimal places...
Rounded Weight (in final encoded dataset).
Rounded Temp_Risk (in final encoded dataset).
Rounded Dangerous_Level (in final encoded dataset).


**Final Data Quality Summary**

In [123]:
print("\n" + "="*80)
print("FINAL PREPROCESSING SUMMARY")
print("="*80)

print("Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

print("\nNumerical Summary:")
print(df.describe())


FINAL PREPROCESSING SUMMARY
Dataset Shape: (59315, 30)
Total Missing Values: 0
Duplicate Rows: 0

Data Types:
Species                   object
Breed                     object
Age                       object
Gender                    object
Weight                   float64
Symptom_1                 object
Symptom_2                 object
Symptom_3                 object
Symptom_4                 object
Duration                  object
Appetite_Loss              int64
Vomiting                   int64
Diarrhea                   int64
Coughing                   int64
Labored_Breathing          int64
Lameness                   int64
Skin_Lesions               int64
Nasal_Discharge            int64
Eye_Discharge              int64
Body_Temperature         float64
Heart_Rate                object
Disease_Prediction        object
Treatment                 object
Description               object
Symptom_Count              int64
Temp_Risk                float64
Symptom_Risk             float6

**Save Cleaned Data**

In [124]:
output_file = "animal_disease_merged_cleaned.csv"
df.to_csv(output_file, index=False)

print("\n✓ Cleaned dataset saved as:", output_file)
print("Rows:", len(df))
print("Columns:", len(df.columns))


✓ Cleaned dataset saved as: animal_disease_merged_cleaned.csv
Rows: 59315
Columns: 30
